## self-sustained activity in a recurrent neural network using snntorch

Some vibe coding produced scripts to learn visualization etc

In [ ]:
import torch
import platform
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print(f'> device (Apple Silicon/MacOS) - macos_version = {platform.mac_ver()[0]}')

elif torch.cuda.is_available():
    device = torch.device('cuda')
    print('Running on GPU : ', torch.cuda.get_device_name(), '#GPU=', torch.cuda.device_count())
else:
    device = torch.device('cpu')
device

: 

In [ ]:
import torch
import torch.nn as nn
import snntorch as snn
import matplotlib.pyplot as plt
import numpy as np


class Net(nn.Module):
    def __init__(self, num_inputs=784, num_hidden=128, num_outputs=10, beta=0.95):
        super().__init__()
        
        # Initialize layers
        self.fc1 = nn.Linear(num_inputs, num_hidden)
        self.lif1 = snn.Leaky(beta=beta)
        self.fc2 = nn.Linear(num_hidden, num_outputs)
        self.lif2 = snn.Leaky(beta=beta)

    def forward(self, x, num_steps=25):
        # Initialize hidden states
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        
        # Record the final layer
        spk1_rec = []
        mem1_rec = []
        spk2_rec = []
        mem2_rec = []
        
        # Simulate for specified time steps
        for _ in range(num_steps):
            cur1 = self.fc1(x)
            spk1, mem1 = self.lif1(cur1, mem1)
            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)
            
            # Record all layers
            spk1_rec.append(spk1)
            mem1_rec.append(mem1)
            spk2_rec.append(spk2)
            mem2_rec.append(mem2)
        
        return (torch.stack(spk1_rec), torch.stack(mem1_rec), 
                torch.stack(spk2_rec), torch.stack(mem2_rec))

# Create and load network
net = Net().to(device)

# Create test input (simulating MNIST-like input)
test_input = torch.rand(1, 784).to(device)

# Run network
spk1_rec, mem1_rec, spk2_rec, mem2_rec = net(test_input, num_steps=500)

print("Running visualization...")

def visualize_snn_activity(spk1_rec, mem1_rec, spk2_rec, mem2_rec, title="SNN Activity Visualization"):
    """Comprehensive visualization of SNN activity"""
    
    # Convert tensors to numpy for plotting
    spk1_np = spk1_rec.detach().cpu().numpy()
    mem1_np = mem1_rec.detach().cpu().numpy()
    spk2_np = spk2_rec.detach().cpu().numpy()
    mem2_np = mem2_rec.detach().cpu().numpy()
    
    time_steps = np.arange(spk1_np.shape[0])
    
    # Create figure with subplots
    fig = plt.figure(figsize=(20, 15))
    fig.suptitle(title, fontsize=16)
    
    # 1. Raster plot of hidden layer spikes (sample subset)
    ax1 = plt.subplot(3, 3, 1)
    sample_neurons = min(50, spk1_np.shape[2])  # Sample up to 50 neurons
    
    spike_times = []
    neuron_ids = []
    
    for neuron_id in range(0, sample_neurons, 5):  # Sample every 5th neuron for clarity
        times_with_spikes = time_steps[spk1_np[:, 0, neuron_id] > 0]
        spike_times.extend(times_with_spikes)
        neuron_ids.extend([neuron_id] * len(times_with_spikes))
    
    ax1.scatter(spike_times, neuron_ids, s=20, alpha=0.7, color='darkorange')
    ax1.set_xlabel('Time Step')
    ax1.set_ylabel('Neuron ID')
    ax1.set_title('Hidden Layer Spike Raster (Sampled)')
    ax1.grid(True, alpha=0.3)
    
    # 2. Membrane potential traces for hidden layer
    ax2 = plt.subplot(3, 3, 2)
    sample_neuron_ids = [0, 10, 20, 30, 40][:min(5, spk1_np.shape[2])]
    
    for neuron_id in sample_neuron_ids:
        ax2.plot(time_steps, mem1_np[:, 0, neuron_id], 
                label=f'Neuron {neuron_id}', linewidth=2, alpha=0.8)
    
    ax2.set_xlabel('Time Step')
    ax2.set_ylabel('Membrane Potential')
    ax2.set_title('Hidden Layer Membrane Potentials')
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)
    
    # 3. Output layer spike activity heatmap
    ax3 = plt.subplot(3, 3, 3)
    im = ax3.imshow(spk2_np[:, 0, :].T, aspect='auto', cmap='hot', interpolation='nearest')
    ax3.set_xlabel('Time Step')
    ax3.set_ylabel('Output Neuron')
    ax3.set_title('Output Layer Spike Activity')
    plt.colorbar(im, ax=ax3, label='Spike (0=No, 1=Yes)')
    
    # 4. Population spike rate over time
    ax4 = plt.subplot(3, 3, 4)
    hidden_spike_rate = np.mean(spk1_np[:, 0, :], axis=1)
    output_spike_rate = np.mean(spk2_np[:, 0, :], axis=1)
    
    ax4.plot(time_steps, hidden_spike_rate, 'orange', linewidth=2, label='Hidden Layer')
    ax4.plot(time_steps, output_spike_rate, 'red', linewidth=2, label='Output Layer')
    ax4.set_xlabel('Time Step')
    ax4.set_ylabel('Average Spike Rate')
    ax4.set_title('Population Spike Rates')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    # 5. Cumulative spike counts
    ax5 = plt.subplot(3, 3, 5)
    hidden_cumsum = np.cumsum(np.sum(spk1_np[:, 0, :], axis=1))
    output_cumsum = np.cumsum(np.sum(spk2_np[:, 0, :], axis=1))
    
    ax5.plot(time_steps, hidden_cumsum, 'orange', linewidth=2, label='Hidden Layer')
    ax5.plot(time_steps, output_cumsum, 'red', linewidth=2, label='Output Layer')
    ax5.set_xlabel('Time Step')
    ax5.set_ylabel('Cumulative Spike Count')
    ax5.set_title('Cumulative Spikes Over Time')
    ax5.legend()
    ax5.grid(True, alpha=0.3)
    
    # 6. Spike count distribution
    ax6 = plt.subplot(3, 3, 6)
    hidden_spike_counts = np.sum(spk1_np[:, 0, :], axis=0)
    output_spike_counts = np.sum(spk2_np[:, 0, :], axis=0)
    
    ax6.hist(hidden_spike_counts, bins=20, alpha=0.7, color='orange', 
             label=f'Hidden (total: {int(np.sum(hidden_spike_counts))})', density=True)
    ax6.hist(output_spike_counts, bins=20, alpha=0.7, color='red', 
             label=f'Output (total: {int(np.sum(output_spike_counts))})', density=True)
    ax6.set_xlabel('Total Spikes per Neuron')
    ax6.set_ylabel('Density')
    ax6.set_title('Spike Count Distribution')
    ax6.legend()
    ax6.grid(True, alpha=0.3)
    
    # 7. Membrane potential statistics for output layer
    ax7 = plt.subplot(3, 3, 7)
    output_max_pot = np.max(mem2_np[:, 0, :], axis=0)
    output_min_pot = np.min(mem2_np[:, 0, :], axis=0)
    
    ax7.scatter(output_min_pot, output_max_pot, alpha=0.7, color='purple')
    ax7.set_xlabel('Minimum Membrane Potential')
    ax7.set_ylabel('Maximum Membrane Potential')
    ax7.set_title('Output Neuron Membrane Potential Range')
    ax7.grid(True, alpha=0.3)
    
    # 8. Time-frequency analysis (spike rate in windows)
    ax8 = plt.subplot(3, 3, 8)
    window_size = 10
    windowed_rates = []
    window_centers = []
    
    for i in range(0, len(time_steps)-window_size, window_size//2):
        rate = np.mean(hidden_spike_rate[i:i+window_size])
        windowed_rates.append(rate)
        window_centers.append(i + window_size//2)
    
    ax8.plot(window_centers, windowed_rates, 'orange', linewidth=2)
    ax8.set_xlabel('Time Window Center')
    ax8.set_ylabel('Average Spike Rate')
    ax8.set_title('Windowed Spike Rate Analysis')
    ax8.grid(True, alpha=0.3)
    
    # 9. Network statistics summary
    ax9 = plt.subplot(3, 3, 9)
    ax9.axis('off')
    
    total_hidden_spikes = np.sum(spk1_np)
    total_output_spikes = np.sum(spk2_np)
    avg_hidden_rate = np.mean(hidden_spike_rate)
    avg_output_rate = np.mean(output_spike_rate)
    sparsity_hidden = np.sum(spk1_np > 0) / spk1_np.size
    sparsity_output = np.sum(spk2_np > 0) / spk2_np.size
    
    stats_text = f"""Network Statistics:
    
    Hidden Layer:
    - Neurons: {spk1_np.shape[2]}
    - Total Spikes: {int(total_hidden_spikes)}
    - Avg Rate: {avg_hidden_rate:.3f}/step
    - Sparsity: {sparsity_hidden*100:.1f}%
    
    Output Layer:
    - Neurons: {spk2_np.shape[2]}
    - Total Spikes: {int(total_output_spikes)}
    - Avg Rate: {avg_output_rate:.3f}/step
    - Sparsity: {sparsity_output*100:.1f}%
    
    Simulation:
    - Time Steps: {len(time_steps)}"""
    
    ax9.text(0.1, 0.9, stats_text, transform=ax9.transAxes, fontsize=10,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
    
    plt.tight_layout()
    plt.show()

def visualize_detailed_temporal(spk1_rec, mem1_rec, spk2_rec, mem2_rec):
    """Detailed temporal visualization focusing on specific time periods"""
    
    # Convert to numpy
    spk1_np = spk1_rec.detach().cpu().numpy()
    mem1_np = mem1_rec.detach().cpu().numpy()
    spk2_np = spk2_rec.detach().cpu().numpy()
    mem2_np = mem2_rec.detach().cpu().numpy()
    
    time_steps = np.arange(spk1_np.shape[0])
    
    # Focus on first 30 time steps for detailed view
    focus_steps = min(30, len(time_steps))
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Detailed Temporal Analysis', fontsize=16)
    
    # 1. Detailed spike raster for hidden layer
    ax1 = axes[0, 0]
    sample_neurons = min(30, spk1_np.shape[2])
    
    spike_times = []
    neuron_ids = []
    
    for neuron_id in range(sample_neurons):
        times_with_spikes = time_steps[:focus_steps][spk1_np[:focus_steps, 0, neuron_id] > 0]
        spike_times.extend(times_with_spikes)
        neuron_ids.extend([neuron_id] * len(times_with_spikes))
    
    ax1.scatter(spike_times, neuron_ids, s=30, alpha=0.7, color='orange')
    ax1.set_xlabel('Time Step')
    ax1.set_ylabel('Neuron ID')
    ax1.set_title('Detailed Hidden Layer Spike Raster')
    ax1.grid(True, alpha=0.3)
    
    # 2. Membrane potential evolution for selected neurons
    ax2 = axes[0, 1]
    selected_neurons = [0, 5, 10, 15, 20][:min(5, spk1_np.shape[2])]
    
    for neuron_id in selected_neurons:
        ax2.plot(time_steps[:focus_steps], mem1_np[:focus_steps, 0, neuron_id], 
                marker='o', markersize=3, label=f'Neuron {neuron_id}')
    
    ax2.set_xlabel('Time Step')
    ax2.set_ylabel('Membrane Potential')
    ax2.set_title('Selected Neuron Membrane Evolution')
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)
    
    # 3. Output layer temporal pattern
    ax3 = axes[1, 0]
    for output_neuron in range(min(5, spk2_np.shape[2])):
        ax3.plot(time_steps[:focus_steps], spk2_np[:focus_steps, 0, output_neuron] * (output_neuron + 1), 
                'o-', markersize=4, alpha=0.7, label=f'Output {output_neuron}')
    
    ax3.set_xlabel('Time Step')
    ax3.set_ylabel('Output Neuron (position indicates ID)')
    ax3.set_title('Output Layer Spike Timing')
    ax3.legend(fontsize=8)
    ax3.grid(True, alpha=0.3)
    
    # 4. Membrane potential traces for output layer
    ax4 = axes[1, 1]
    for output_neuron in range(min(5, mem2_np.shape[2])):
        ax4.plot(time_steps[:focus_steps], mem2_np[:focus_steps, 0, output_neuron], 
                linewidth=2, label=f'Output {output_neuron}')
    
    ax4.set_xlabel('Time Step')
    ax4.set_ylabel('Membrane Potential')
    ax4.set_title('Output Layer Membrane Potentials')
    ax4.legend(fontsize=8)
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Run visualizations
visualize_snn_activity(spk1_rec, mem1_rec, spk2_rec, mem2_rec, 
                      "Basic SNN Activity Visualization")

visualize_detailed_temporal(spk1_rec, mem1_rec, spk2_rec, mem2_rec)

# Print some statistics
print("\n=== Network Activity Statistics ===")
spk1_np = spk1_rec.detach().cpu().numpy()
spk2_np = spk2_rec.detach().cpu().numpy()

print(f"Hidden layer neurons: {spk1_np.shape[2]}")
print(f"Output layer neurons: {spk2_np.shape[2]}")
print(f"Time steps simulated: {spk1_np.shape[0]}")
print(f"Total hidden spikes: {int(np.sum(spk1_np))}")
print(f"Total output spikes: {int(np.sum(spk2_np))}")
print(f"Hidden layer sparsity: {np.sum(spk1_np > 0) / spk1_np.size * 100:.2f}%")
print(f"Output layer sparsity: {np.sum(spk2_np > 0) / spk2_np.size * 100:.2f}%")
